# Qwen3.5-9B Tool-Calling GSPO 训练

**阶段 2**: 在 SFT 模型基础上，用 GSPO (Group Sequence Policy Optimization) 强化学习优化工具调用准确性。

- **SFT** 教会了模型工具调用的格式和模式
- **GSPO** 让模型通过自我探索，学习选择正确的工具和参数

GSPO 是阿里 Qwen 团队提出的 GRPO 改进版 ([arXiv:2507.18071](https://arxiv.org/abs/2507.18071))，
将重要性采样从 token 级改到 sequence 级，训练更稳定。

---

**前置条件**: 已完成 SFT 训练，模型已推送到 `icecee/Qwen3.5-9B-Tool-Calling`

## 1. 安装依赖

In [ ]:
%%capture
!pip install unsloth vllm
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. GPU 检测

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# GRPO 比 SFT 吃更多显存 (要生成多个候选)，必须用 4-bit
if vram_gb >= 35:
    GRPO_CONFIG = {
        "num_generations": 4,
        "batch_size": 2,
        "grad_accum": 4,
        "max_completion_length": 512,
        "max_prompt_length": 1024,
        "lora_r": 16,
    }
    print("→ A100 模式: 4-bit QLoRA, 4 generations, batch=2")
elif vram_gb >= 20:
    GRPO_CONFIG = {
        "num_generations": 4,
        "batch_size": 1,
        "grad_accum": 8,
        "max_completion_length": 512,
        "max_prompt_length": 512,
        "lora_r": 8,
    }
    print("→ L4/A10 模式: 4-bit QLoRA, 4 generations, batch=1")
else:
    GRPO_CONFIG = {
        "num_generations": 2,
        "batch_size": 1,
        "grad_accum": 8,
        "max_completion_length": 256,
        "max_prompt_length": 512,
        "lora_r": 8,
    }
    print("→ T4 模式: 4-bit QLoRA, 2 generations, batch=1")

print(f"\n配置: {GRPO_CONFIG}")

## 3. 加载 SFT 模型

In [ ]:
from unsloth import FastLanguageModel
import gc

if 'model' in dir(): del model
if 'processor' in dir(): del processor
gc.collect()
torch.cuda.empty_cache()

model, processor = FastLanguageModel.from_pretrained(
    model_name="icecee/Qwen3.5-9B-Tool-Calling",  # SFT 后的模型
    max_seq_length=2048,
    load_in_4bit=True,      # GRPO 必须 4-bit，生成阶段很吃显存
    fast_inference=True,    # vLLM 加速生成
    gpu_memory_utilization=0.6,
)

if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer
    print(f'从 processor 中提取 tokenizer: {type(tokenizer).__name__}')
else:
    tokenizer = processor

print(f"模型加载完成: {type(model).__name__}")
print(f"has chat_template: {tokenizer.chat_template is not None}")
print(f"VRAM 使用: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. 配置 LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=GRPO_CONFIG["lora_r"],
    lora_alpha=GRPO_CONFIG["lora_r"],
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## 5. 准备 GRPO 数据

从 SFT 训练数据中提取有工具调用的样本，转为 GRPO 格式。

GRPO 只需要 prompt（模型自己生成回答），加上 expected_tool/expected_args 用于计算 reward。

In [ ]:
import json, hashlib, random, os
from datasets import Dataset

#@title 选择数据加载方式 { run: "auto" }
DATA_SOURCE = "download_fresh"  #@param ["upload_local", "download_fresh"]

os.makedirs("/content/data", exist_ok=True)

if DATA_SOURCE == "upload_local":
    from google.colab import files
    print("请上传 grpo_train.jsonl (或 train.jsonl，会自动提取)")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f"/content/data/{name}", "wb") as f:
            f.write(content)

# 如果直接有 grpo_train.jsonl 就用，否则从 SFT 数据中提取
grpo_path = "/content/data/grpo_train.jsonl"
sft_path = "/content/data/train.jsonl"

if not os.path.exists(grpo_path):
    if not os.path.exists(sft_path) and DATA_SOURCE == "download_fresh":
        # 重新下载 SFT 数据集并转换 (复用 SFT notebook 的数据准备逻辑)
        print("下载数据集...")
        from datasets import load_dataset as _ld
        
        all_samples = []
        
        def wrap_tool(t):
            if "type" in t and t["type"]=="function" and "function" in t: return t
            func = {"name":t.get("name","?"),"description":t.get("description","")}
            func["parameters"] = t.get("parameters",{"type":"object","properties":{}})
            return {"type":"function","function":func}
        
        def make_tc(name, args):
            if isinstance(args,str):
                try: args=json.loads(args)
                except: args={}
            if not isinstance(args,dict): args={}
            return {"type":"function","function":{"name":name,"arguments":args}}
        
        def convert_sharegpt(ds, lang="en"):
            results=[]
            sys_msg="You are a helpful assistant with access to tools."
            for row in ds:
                try:
                    convs=row.get("conversations",[])
                    ts=row.get("tools","[]")
                    tools=[wrap_tool(t) for t in (json.loads(ts) if isinstance(ts,str) else ts) if isinstance(t,dict)] if ts else []
                    if not tools: continue
                    msgs=[{"role":"system","content":sys_msg}]
                    for c in convs:
                        rf,val=c.get("from",""),c.get("value","")
                        if rf=="human": msgs.append({"role":"user","content":val})
                        elif rf=="gpt": msgs.append({"role":"assistant","content":val})
                        elif rf=="function_call":
                            try:
                                fc=json.loads(val) if isinstance(val,str) else val
                                msgs.append({"role":"assistant","content":None,"tool_calls":[make_tc(fc.get("name","?"),fc.get("arguments",{}))]})
                            except: msgs.append({"role":"assistant","content":val})
                    roles=[m["role"] for m in msgs]
                    if "user" in roles and "assistant" in roles:
                        s={"messages":msgs}
                        if tools: s["tools"]=tools
                        results.append(s)
                except: continue
            return results
        
        # 只下载含工具调用的数据集
        print("[1/3] glaive_v2_sharegpt...")
        ds=_ld("hiyouga/glaive-function-calling-v2-sharegpt",split="train")
        all_samples.extend(convert_sharegpt(ds))
        print(f"  {len(all_samples)} samples")
        
        print("[2/3] glaive_zh...")
        ds=_ld("llamafactory/glaive_toolcall_zh",split="train")
        all_samples.extend(convert_sharegpt(ds,"zh"))
        print(f"  total: {len(all_samples)}")
        
        print("[3/3] Deepexi...")
        ds=_ld("Deepexi/function-calling-small",split="train")
        for row in ds:
            try:
                up,ar=row.get("userPrompt",""),row.get("assistantResponse","")
                if not up or not ar: continue
                resp=json.loads(ar)
                if isinstance(resp,dict) and "function" in resp:
                    args=resp.get("arguments",{})
                    if isinstance(args,list) and args: args=args[0] if isinstance(args[0],dict) else {}
                    tc=[make_tc(resp["function"],args)]
                    msgs=[{"role":"system","content":"你是一个有用的助手。"},{"role":"user","content":up},{"role":"assistant","content":None,"tool_calls":tc}]
                    all_samples.append({"messages":msgs})
            except: continue
        print(f"  total: {len(all_samples)}")
        
        # 写入 SFT 格式
        with open(sft_path,"w") as f:
            for s in all_samples:
                f.write(json.dumps(s,ensure_ascii=False)+"\n")
        print(f"SFT 数据已保存: {len(all_samples)} 样本")

    # 从 SFT 数据提取 GRPO 格式
    print("\n提取 GRPO 数据...")
    samples = []
    with open(sft_path) as f:
        for line in f:
            try:
                row = json.loads(line)
                msgs = row.get("messages",[])
                tools = row.get("tools")
                if not tools: continue
                prompt_msgs = []
                expected_tool = None
                expected_args = {}
                for m in msgs:
                    role = m.get("role")
                    if role == "system":
                        prompt_msgs.append({"role":"system","content":m.get("content","")})
                    elif role == "user" and not any(p["role"]=="user" for p in prompt_msgs):
                        prompt_msgs.append({"role":"user","content":m.get("content","")})
                    elif role == "assistant" and m.get("tool_calls"):
                        tc = m["tool_calls"][0]
                        func = tc.get("function",tc)
                        expected_tool = func.get("name","")
                        ea = func.get("arguments",{})
                        if isinstance(ea,str):
                            try: ea=json.loads(ea)
                            except: ea={}
                        expected_args = ea if isinstance(ea,dict) else {}
                        break
                if not prompt_msgs or not expected_tool: continue
                if not any(m["role"]=="user" for m in prompt_msgs): continue
                if not any(m["role"]=="system" for m in prompt_msgs):
                    prompt_msgs.insert(0,{"role":"system","content":"You are a helpful assistant with access to tools."})
                samples.append({"prompt":prompt_msgs,"tools":tools,"expected_tool":expected_tool,"expected_args":expected_args})
            except: continue
    
    # 去重 + 采样
    seen = set()
    deduped = []
    for s in samples:
        uc = next((m["content"] for m in s["prompt"] if m["role"]=="user"),"")
        h = hashlib.md5(uc.encode()).hexdigest()
        if h not in seen: seen.add(h); deduped.append(s)
    random.seed(42)
    random.shuffle(deduped)
    final = deduped[:8000]
    
    with open(grpo_path,"w") as f:
        for s in final:
            f.write(json.dumps(s,ensure_ascii=False)+"\n")
    print(f"GRPO 数据已保存: {len(final)} 样本")

else:
    print(f"GRPO 数据已存在: {grpo_path}")

In [ ]:
# 加载 GRPO 数据为 Dataset
import json
from datasets import Dataset

rows = []
with open("/content/data/grpo_train.jsonl") as f:
    for line in f:
        row = json.loads(line)
        # GRPOTrainer 需要 prompt 列，格式为 apply_chat_template 能处理的 messages
        # 把 tools 信息编入 system prompt 中 (通过 chat_template)
        prompt_text = tokenizer.apply_chat_template(
            row["prompt"],
            tools=row.get("tools"),
            tokenize=False,
            add_generation_prompt=True,
        )
        rows.append({
            "prompt": prompt_text,
            "expected_tool": row["expected_tool"],
            "expected_args": json.dumps(row.get("expected_args", {}), ensure_ascii=False),
        })

grpo_dataset = Dataset.from_list(rows)
print(f"GRPO 数据集: {len(grpo_dataset)} 样本")
print(f"示例 prompt (前 200 字符): {grpo_dataset[0]['prompt'][:200]}")

## 6. 定义 Reward 函数

3 个 reward 函数，加权组合：
- **tool_selection_reward** (权重 2.0): 选对了工具
- **format_reward** (权重 1.0): 输出格式合规
- **args_reward** (权重 1.5): 参数匹配度

In [ ]:
import re

def tool_selection_reward(completions, expected_tool, **kwargs):
    """选对工具 +1.0, 调了但调错 -0.5, 没调用 0.0"""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        if not isinstance(text, str): text = str(text)
        if f"<function={expected_tool[0]}>" in text:
            rewards.append(1.0)
        elif "<tool_call>" in text or "<function=" in text:
            rewards.append(-0.5)  # 调了但调错了
        else:
            rewards.append(0.0)   # 没调用工具
    return rewards


def format_reward(completions, **kwargs):
    """输出包含合法的 tool_call 格式 +1.0, 格式错误但尝试了 +0.3, 没有 0.0"""
    valid_pattern = re.compile(r"<tool_call>\s*<function=\w[\w.]*>")
    partial_pattern = re.compile(r"<tool_call>|<function=")
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        if not isinstance(text, str): text = str(text)
        if valid_pattern.search(text):
            rewards.append(1.0)
        elif partial_pattern.search(text):
            rewards.append(0.3)
        else:
            rewards.append(0.0)
    return rewards


def args_reward(completions, expected_args, **kwargs):
    """参数名匹配度: 全匹配 1.0, 部分匹配 0.x, 无匹配 0.0"""
    rewards = []
    for i, c in enumerate(completions):
        text = c[0]["content"] if isinstance(c, list) else c
        if not isinstance(text, str): text = str(text)
        # 解析期望的参数 key
        ea = expected_args[i] if isinstance(expected_args, list) else expected_args
        if isinstance(ea, str):
            try: ea = json.loads(ea)
            except: ea = {}
        expected_keys = set(ea.keys()) if isinstance(ea, dict) else set()
        if not expected_keys:
            rewards.append(0.5)  # 没有期望参数，给中性分
            continue
        # 从模型输出中提取 <parameter=key>
        found_params = set(re.findall(r"<parameter=(\w+)>", text))
        if not found_params:
            rewards.append(0.0)
        else:
            overlap = len(found_params & expected_keys) / len(expected_keys)
            rewards.append(overlap)
    return rewards


# 测试 reward 函数
test_good = ["<tool_call>\n<function=get_weather>\n<parameter=city>\nBeijing\n</parameter>\n</function>\n</tool_call>"]
test_bad = ["I don't know how to help with that."]
test_wrong = ["<tool_call>\n<function=wrong_tool>\n</function>\n</tool_call>"]

print("Reward 测试:")
print(f"  正确调用: tool={tool_selection_reward(test_good, ['get_weather'])}, fmt={format_reward(test_good)}, args={args_reward(test_good, ['{\"city\": \"Beijing\"}'])}")
print(f"  未调用:   tool={tool_selection_reward(test_bad, ['get_weather'])}, fmt={format_reward(test_bad)}, args={args_reward(test_bad, ['{\"city\": \"Beijing\"}'])}")
print(f"  调错工具: tool={tool_selection_reward(test_wrong, ['get_weather'])}, fmt={format_reward(test_wrong)}, args={args_reward(test_wrong, ['{\"city\": \"Beijing\"}'])}")

## 7. GSPO 训练

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import torch.fx.experimental._config as fx_config
fx_config.meta_nonzero_assume_all_nonzero = True

training_args = GRPOConfig(
    output_dir="/content/grpo_output",
    
    # 生成参数
    num_generations=GRPO_CONFIG["num_generations"],
    max_completion_length=GRPO_CONFIG["max_completion_length"],
    max_prompt_length=GRPO_CONFIG["max_prompt_length"],
    
    # 训练参数
    per_device_train_batch_size=GRPO_CONFIG["batch_size"],
    gradient_accumulation_steps=GRPO_CONFIG["grad_accum"],
    max_steps=500,
    learning_rate=5e-6,         # RL 用比 SFT 低的学习率
    warmup_steps=30,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    
    # GSPO 核心配置
    beta=0.0,                   # 不用 KL 约束
    loss_type="dr_grpo",        # 去长度偏差
    importance_sampling_level="sequence",  # ← GSPO: 序列级重要性采样
    
    # 精度和优化
    bf16=True,
    optim="adamw_8bit",
    torch_compile=True,
    
    # 日志和保存
    logging_steps=5,
    save_steps=500,
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[tool_selection_reward, format_reward, args_reward],
    reward_weights=[2.0, 1.0, 1.5],
    train_dataset=grpo_dataset,
    args=training_args,
)

print(f"GSPO 配置:")
print(f"  每 prompt 生成 {GRPO_CONFIG['num_generations']} 个候选")
print(f"  有效 batch: {GRPO_CONFIG['batch_size'] * GRPO_CONFIG['grad_accum']}")
print(f"  loss_type: dr_grpo + sequence-level importance sampling (GSPO)")
print(f"  reward: tool_selection(2.0) + format(1.0) + args(1.5)")
print(f"  总步数: 500")
print("\n开始 GSPO 训练...\n")

trainer.train()

## 8. 保存模型

In [ ]:
# 保存 LoRA 适配器
model.save_pretrained("/content/grpo_output/lora_adapter")
processor.save_pretrained("/content/grpo_output/lora_adapter")
print("GSPO LoRA 适配器已保存")

# 保存合并后的完整模型
model.save_pretrained_merged(
    "/content/grpo_output/merged_bf16",
    processor,
    save_method="merged_16bit",
)
print("合并模型 (bf16) 已保存")

## 9. 测试工具调用 (GSPO 后)

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    {
        "name": "中文-天气查询",
        "messages": [{"role": "user", "content": "请帮我查询北京今天的天气"}],
        "tools": [{"type": "function", "function": {
            "name": "get_weather", "description": "查询指定城市天气",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
        }}],
    },
    {
        "name": "EN-Calculator",
        "messages": [{"role": "user", "content": "Calculate 15% tip on a $85 dinner bill"}],
        "tools": [{"type": "function", "function": {
            "name": "calculate_tip", "description": "Calculate tip amount",
            "parameters": {"type": "object", "properties": {
                "bill_amount": {"type": "number"}, "tip_percentage": {"type": "number"}
            }, "required": ["bill_amount", "tip_percentage"]}
        }}],
    },
    {
        "name": "中文-多工具选择",
        "messages": [{"role": "user", "content": "帮我搜索一下关于Python的书籍"}],
        "tools": [
            {"type": "function", "function": {
                "name": "search_books", "description": "搜索书籍",
                "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
            }},
            {"type": "function", "function": {
                "name": "search_movies", "description": "搜索电影",
                "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
            }},
        ],
    },
    {
        "name": "EN-No tool needed",
        "messages": [{"role": "user", "content": "Hello, how are you?"}],
        "tools": None,
    },
]

for tc in test_cases:
    print(f"\n{'='*50}")
    print(f"测试: {tc['name']}")
    print(f"User: {tc['messages'][0]['content']}")
    if tc.get('tools'): print(f"Tools: {[t['function']['name'] for t in tc['tools']]}")

    input_text = tokenizer.apply_chat_template(
        tc["messages"], tools=tc["tools"], tokenize=False, add_generation_prompt=True
    )
    inputs = processor(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    if "<|im_end|>" in resp: resp = resp[:resp.index("<|im_end|")]
    print(f"\nAssistant: {resp.strip()}")

## 10. 保存到 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/qwen35-tool-calling-gspo
!cp -r /content/grpo_output/lora_adapter /content/drive/MyDrive/qwen35-tool-calling-gspo/

print("模型已保存到 Google Drive: /MyDrive/qwen35-tool-calling-gspo/")

## 11. 推送到 HuggingFace Hub (可选)

In [ ]:
# HF_TOKEN = "hf_xxx"
# HF_REPO = "icecee/Qwen3.5-9B-Tool-Calling-GSPO"

# model.push_to_hub_merged(HF_REPO, processor, save_method="merged_16bit", token=HF_TOKEN)
# print(f"已推送到: https://huggingface.co/{HF_REPO}")